# 02 - Data Merge & Exploration

1. Load HLS CSVs from GEE (L8 + S2, both approaches)
2. Merge into unified time series per field
3. Join with phenology ground truth
4. Exploratory analysis & visualization

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import os
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# === PATHS ===
# GEE exports
GEE_DIR = 'data/raw/satellite'

# Phenology data
PHENO_EXACT = 'data/processed/exact/wheat_hrw_phenology_csb_matched.parquet'
PHENO_BUFFER = 'data/processed/buffer_300m/wheat_hrw_phenology_buffer_matched.parquet'

# Output
OUTPUT_DIR = 'data/processed'

# Approach config: field_id is the column name used in each CSV
APPROACHES = {
    # CSB-exact match approach removed; using 300m buffer approach exclusively.
    'buffer': {'l8': 'buffer_l8_timeseries.csv', 's2': 'buffer_s2_timeseries.csv',
               'pheno': PHENO_BUFFER, 'field_id': 'FIELDID'}
}

## 1. Load & Merge HLS Data

In [ ]:
def load_and_merge_hls(approach_name, file_info):
    """Load L8 and S2 CSVs and merge into unified time series."""
    fid = file_info['field_id']
    
    print(f'\n{"="*60}')
    print(f'Loading {approach_name.upper()} approach (field_id={fid})')
    print(f'{"="*60}')
    
    df_l8 = pd.read_csv(os.path.join(GEE_DIR, file_info['l8']))
    print(f'L8: {len(df_l8):,} rows, {df_l8[fid].nunique()} fields')
    
    df_s2 = pd.read_csv(os.path.join(GEE_DIR, file_info['s2']))
    print(f'S2: {len(df_s2):,} rows, {df_s2[fid].nunique()} fields')
    
    df_l8 = df_l8.rename(columns={fid: 'field_id'})
    df_s2 = df_s2.rename(columns={fid: 'field_id'})
    
    df = pd.concat([df_l8, df_s2], ignore_index=True)
    df['date'] = pd.to_datetime(df['date'])
    df['year'] = df['date'].dt.year
    df['month'] = df['date'].dt.month
    df['doy'] = df['date'].dt.dayofyear
    df['field_id'] = df['field_id'].astype(str)
    df = df.sort_values(['field_id', 'date']).reset_index(drop=True)
    
    # FIX: GEE double-scaled bands (0.0001 applied twice)
    # Restore correct reflectance (0-1 range)
    band_cols = ['Blue', 'Green', 'Red', 'NIR', 'SWIR1', 'SWIR2']
    re_cols = ['RE1', 'RE2', 'RE3']
    for col in band_cols + re_cols:
        if col in df.columns:
            df[col] = df[col] * 10000
    
    # Recompute ALL indices from corrected bands
    df['NDVI'] = (df['NIR'] - df['Red']) / (df['NIR'] + df['Red'])
    df['EVI'] = 2.5 * (df['NIR'] - df['Red']) / (df['NIR'] + 6*df['Red'] - 7.5*df['Blue'] + 1)
    df['GCVI'] = df['NIR'] / df['Green'] - 1
    
    if 'RE1' in df.columns:
        mask_s2 = df['RE1'].notna()
        df.loc[mask_s2, 'NDRE'] = (df.loc[mask_s2, 'NIR'] - df.loc[mask_s2, 'RE1']) / (df.loc[mask_s2, 'NIR'] + df.loc[mask_s2, 'RE1'])
        df.loc[mask_s2, 'CIre'] = df.loc[mask_s2, 'NIR'] / df.loc[mask_s2, 'RE1'] - 1
        df.loc[mask_s2, 'MTCI'] = (df.loc[mask_s2, 'RE2'] - df.loc[mask_s2, 'RE1']) / (df.loc[mask_s2, 'RE1'] - df.loc[mask_s2, 'Red'])
    
    print(f'\nCombined: {len(df):,} rows, {df["field_id"].nunique():,} fields')
    print(f'Date range: {df["date"].min().date()} to {df["date"].max().date()}')
    print(f'Sensor: {df["sensor"].value_counts().to_dict()}')
    print(f'\nBand/VI check (corrected):')
    for col in ['Blue', 'Red', 'NIR', 'NDVI', 'EVI', 'GCVI']:
        print(f'  {col}: mean={df[col].mean():.4f}')
    
    return df

dfs = {}
for approach, info in APPROACHES.items():
    dfs[approach] = load_and_merge_hls(approach, info)

In [ ]:
# Quick data quality check
for approach, df in dfs.items():
    print(f'\n--- {approach.upper()} ---')
    vi_cols = ['NDVI', 'EVI', 'GCVI']
    if 'NDRE' in df.columns:
        vi_cols += ['NDRE', 'CIre', 'MTCI']
    
    for col in vi_cols:
        if col in df.columns:
            valid = df[col].dropna()
            print(f'  {col}: n={len(valid):,}, mean={valid.mean():.4f}, '
                  f'min={valid.min():.4f}, max={valid.max():.4f}')
    
    # Check for unreasonable values
    if 'NDVI' in df.columns:
        bad = ((df['NDVI'] < -0.5) | (df['NDVI'] > 1.0)).sum()
        print(f'  NDVI out of [-0.5, 1.0]: {bad}')

## 2. Join with Phenology Ground Truth

In [ ]:
def join_phenology(df_hls, pheno_path, approach_name, orig_field_id):
    """Join HLS time series with phenology observations."""
    df_pheno = pd.read_parquet(pheno_path)
    
    # Standardize field ID in phenology data
    if orig_field_id in df_pheno.columns:
        df_pheno = df_pheno.rename(columns={orig_field_id: 'field_id'})
    df_pheno['field_id'] = df_pheno['field_id'].astype(str)
    
    # Get flag leaf dates per field per year
    flag_leaf = df_pheno[df_pheno['growth_stage'].isin(['Flag Leaf Emerging', 'Flag Leaf Emerged'])].copy()
    
    flag_dates = flag_leaf.groupby(['field_id', 'year', 'growth_stage']).agg(
        flag_doy=('doy', 'median'),
        flag_date=('date', 'min')
    ).reset_index()
    
    # Pivot: one row per field+year with both stages
    flag_wide = flag_dates.pivot_table(
        index=['field_id', 'year'],
        columns='growth_stage',
        values='flag_doy'
    ).reset_index()
    flag_wide.columns.name = None
    flag_wide = flag_wide.rename(columns={
        'Flag Leaf Emerging': 'flag_emerging_doy',
        'Flag Leaf Emerged': 'flag_emerged_doy'
    })
    
    # Get all growth stages per field per year
    all_stages = df_pheno.groupby(['field_id', 'year']).agg(
        stages=('growth_stage', lambda x: list(x.unique())),
        n_obs=('date', 'count'),
        lat=('lat', 'median'),
        lon=('lon', 'median'),
    ).reset_index()
    
    # Merge
    field_info = all_stages.merge(flag_wide, on=['field_id', 'year'], how='left')
    
    df_merged = df_hls.merge(
        field_info[['field_id', 'year', 'lat', 'lon', 'flag_emerging_doy', 'flag_emerged_doy']],
        on=['field_id', 'year'],
        how='left'
    )
    
    has_flag = df_merged['flag_emerging_doy'].notna()
    print(f'\n{approach_name}: {has_flag.sum():,} / {len(df_merged):,} HLS obs have flag leaf ground truth')
    print(f'Fields with flag leaf data: {df_merged[has_flag]["field_id"].nunique()}')
    
    return df_merged, field_info

merged = {}
field_infos = {}
for approach, info in APPROACHES.items():
    merged[approach], field_infos[approach] = join_phenology(
        dfs[approach], info['pheno'], approach, info['field_id'])

## 3. Time Series Visualization

In [ ]:
# Plot sample time series for fields WITH flag leaf data
def plot_field_timeseries(df, field_info, approach_name, n_fields=6):
    """Plot VI time series for sample fields with flag leaf dates."""
    fields_with_flag = field_info[field_info['flag_emerging_doy'].notna()]
    sample_fields = fields_with_flag.sample(min(n_fields, len(fields_with_flag)), random_state=42)
    
    fig, axes = plt.subplots(n_fields, 1, figsize=(14, 3*n_fields), sharex=False)
    if n_fields == 1:
        axes = [axes]
    
    vis = ['NDVI', 'EVI', 'GCVI']
    colors = {'NDVI': 'green', 'EVI': 'blue', 'GCVI': 'orange'}
    
    for i, (_, field) in enumerate(sample_fields.iterrows()):
        ax = axes[i]
        fid = field['field_id']
        year = field['year']
        
        sub = df[(df['field_id'] == fid) & (df['year'] == year)].copy()
        
        for vi in vis:
            if vi in sub.columns:
                valid = sub[sub[vi].notna()]
                l8 = valid[valid['sensor'] == 'L8']
                s2 = valid[valid['sensor'] == 'S2']
                if len(l8) > 0:
                    ax.scatter(l8['doy'], l8[vi], c=colors[vi], s=20, marker='o', alpha=0.7)
                if len(s2) > 0:
                    ax.scatter(s2['doy'], s2[vi], c=colors[vi], s=20, marker='^', alpha=0.7)
                ax.plot(valid['doy'], valid[vi], c=colors[vi], alpha=0.3, label=vi)
        
        if pd.notna(field.get('flag_emerging_doy')):
            ax.axvline(field['flag_emerging_doy'], color='red', linestyle='--', alpha=0.8, label='Flag Emerging')
        if pd.notna(field.get('flag_emerged_doy')):
            ax.axvline(field['flag_emerged_doy'], color='darkred', linestyle='-', alpha=0.8, label='Flag Emerged')
        
        ax.set_ylabel('VI value')
        ax.set_title(f'{fid} | {year} | lat={field["lat"]:.2f}', fontsize=10)
        if i == 0:
            ax.legend(loc='upper right', fontsize=8)
        ax.set_xlim(1, 366)
    
    axes[-1].set_xlabel('Day of Year')
    fig.suptitle(f'{approach_name.upper()} — Sample Field Time Series', fontsize=14, y=1.01)
    plt.tight_layout()
    plt.show()

for approach in merged:
    plot_field_timeseries(merged[approach], field_infos[approach], approach)

In [ ]:
# Red-edge indices (S2 only) for fields with flag leaf data
def plot_rededge_timeseries(df, field_info, n_fields=4):
    s2_data = df[df['sensor'] == 'S2'].copy()
    if len(s2_data) == 0 or 'NDRE' not in s2_data.columns:
        print('No S2 red-edge data available')
        return
    
    fields_with_flag = field_info[
        (field_info['flag_emerging_doy'].notna()) & 
        (field_info['year'] >= 2016)
    ]
    sample = fields_with_flag.sample(min(n_fields, len(fields_with_flag)), random_state=99)
    
    fig, axes = plt.subplots(n_fields, 1, figsize=(14, 3*n_fields))
    if n_fields == 1:
        axes = [axes]
    
    re_vis = ['NDRE', 'CIre', 'MTCI']
    colors = {'NDRE': 'red', 'CIre': 'purple', 'MTCI': 'brown'}
    
    for i, (_, field) in enumerate(sample.iterrows()):
        ax = axes[i]
        sub = s2_data[(s2_data['field_id'] == field['field_id']) & (s2_data['year'] == field['year'])]
        
        for vi in re_vis:
            if vi in sub.columns:
                valid = sub[sub[vi].notna()]
                ax.plot(valid['doy'], valid[vi], 'o-', markersize=4, label=vi, color=colors[vi], alpha=0.7)
        
        if pd.notna(field.get('flag_emerging_doy')):
            ax.axvline(field['flag_emerging_doy'], color='red', linestyle='--', alpha=0.8)
        if pd.notna(field.get('flag_emerged_doy')):
            ax.axvline(field['flag_emerged_doy'], color='darkred', linestyle='-', alpha=0.8)
        
        ax.set_ylabel('VI value')
        ax.set_title(f'{field["field_id"]} | {field["year"]} | lat={field["lat"]:.2f}', fontsize=10)
        if i == 0:
            ax.legend(fontsize=8)
    
    axes[-1].set_xlabel('Day of Year')
    fig.suptitle('Red-Edge Indices (Sentinel-2 only)', fontsize=14, y=1.01)
    plt.tight_layout()
    plt.show()

plot_rededge_timeseries(merged['buffer'], field_infos['buffer'])

## 4. Temporal Density Analysis

In [ ]:
# How many clean observations per field per year?
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for i, (approach, df) in enumerate(merged.items()):
    obs_per_field_year = df.groupby(['field_id', 'year']).size().reset_index(name='n_obs')
    
    for year in sorted(df['year'].unique()):
        sub = obs_per_field_year[obs_per_field_year['year'] == year]
        axes[i].hist(sub['n_obs'], bins=30, alpha=0.5, label=str(year))
    
    axes[i].set_xlabel('Clean observations per field per year')
    axes[i].set_ylabel('Count')
    axes[i].set_title(f'{approach.upper()}')
    axes[i].legend()
    axes[i].axvline(obs_per_field_year['n_obs'].median(), color='red', linestyle='--')

plt.suptitle('Temporal Density of Clean Observations', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Temporal gap analysis around flag leaf period (DOY 80-150)
for approach, df in merged.items():
    print(f'\n--- {approach.upper()} ---')
    spring = df[(df['doy'] >= 80) & (df['doy'] <= 150)]
    obs_spring = spring.groupby(['field_id', 'year']).size()
    print(f'Obs per field in flag leaf window (DOY 80-150):')
    print(f'  Mean: {obs_spring.mean():.1f}, Median: {obs_spring.median():.0f}')
    print(f'  Min: {obs_spring.min()}, Max: {obs_spring.max()}')
    
    gaps = spring.groupby(['field_id', 'year'])['doy'].apply(
        lambda x: x.sort_values().diff().dropna().mean()
    )
    print(f'  Mean gap between obs: {gaps.mean():.1f} days')

## 5. VI at Flag Leaf Stage

In [ ]:
# What VI values do we see at/near flag leaf dates?
def vi_at_flag_leaf(df, field_info, approach_name):
    fields_flag = field_info[field_info['flag_emerging_doy'].notna()].copy()
    
    results = []
    for _, field in fields_flag.iterrows():
        sub = df[(df['field_id'] == field['field_id']) & (df['year'] == field['year'])].copy()
        if len(sub) == 0:
            continue
        
        flag_doy = field['flag_emerging_doy']
        sub['days_from_flag'] = sub['doy'] - flag_doy
        
        nearby = sub[sub['days_from_flag'].abs() <= 10]
        if len(nearby) == 0:
            continue
        
        closest = nearby.loc[nearby['days_from_flag'].abs().idxmin()]
        results.append({
            'field_id': field['field_id'],
            'year': field['year'],
            'lat': field['lat'],
            'flag_doy': flag_doy,
            'obs_doy': closest['doy'],
            'days_off': closest['days_from_flag'],
            'sensor': closest['sensor'],
            'NDVI': closest.get('NDVI'),
            'EVI': closest.get('EVI'),
            'GCVI': closest.get('GCVI'),
            'NDRE': closest.get('NDRE'),
            'CIre': closest.get('CIre'),
            'MTCI': closest.get('MTCI'),
        })
    
    df_flag_vi = pd.DataFrame(results)
    print(f'\n{approach_name}: {len(df_flag_vi)} fields with VI near flag leaf')
    
    if len(df_flag_vi) > 0:
        print(f'\nVI values at flag leaf emergence (+/- 10 days):')
        for vi in ['NDVI', 'EVI', 'GCVI', 'NDRE', 'CIre', 'MTCI']:
            if vi in df_flag_vi.columns:
                vals = df_flag_vi[vi].dropna()
                if len(vals) > 0:
                    print(f'  {vi}: mean={vals.mean():.4f}, std={vals.std():.4f}, n={len(vals)}')
    
    return df_flag_vi

flag_vi = {}
for approach in merged:
    flag_vi[approach] = vi_at_flag_leaf(merged[approach], field_infos[approach], approach)

In [ ]:
# Plot VI distributions at flag leaf stage
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
vis = ['NDVI', 'EVI', 'GCVI', 'NDRE', 'CIre', 'MTCI']

for i, vi in enumerate(vis):
    ax = axes[i//3, i%3]
    for approach, df_fv in flag_vi.items():
        if vi in df_fv.columns:
            vals = df_fv[vi].dropna()
            if len(vals) > 5:
                ax.hist(vals, bins=25, alpha=0.5, label=f'{approach} (n={len(vals)})')
    ax.set_title(vi)
    ax.legend(fontsize=8)

plt.suptitle('VI Distribution at Flag Leaf Emergence (+/- 10 days)', fontsize=14)
plt.tight_layout()
plt.show()

## 6. Save Merged Datasets

In [ ]:
for approach, df in merged.items():
    out_dir = os.path.join(OUTPUT_DIR, approach)
    os.makedirs(out_dir, exist_ok=True)
    out_path = os.path.join(out_dir, 'hls_phenology_merged.parquet')
    df.to_parquet(out_path, index=False)
    print(f'{approach}: saved {len(df):,} rows to {out_path}')

print('\nDone! Ready for notebook 03 (flag leaf detection).')